# Eval MoE v2 — Fixed Evaluation

**Use this after running `train_moe_sprint3_v2.ipynb` and saving the checkpoint output as a Kaggle dataset.**

## Fixes vs the broken eval
| # | Fix | Why |
|---|---|---|
| E1 | Checkpoint paths updated to `mabdullahi454/tb-pipeline-checkpoints` | Old notebook pointed at `iahmedhabib` slug which doesn't exist |
| E2 | Bootstrap 95% CI on Montgomery Timika AUROC | n=28, 14 TB+ → ±0.20 variance; raw number is meaningless without CI |
| E3 | NIH ALP sanity gate | If NIH ALP mean > 8%, experts are still painting lungs |
| E4 | TBX11K bbox diagnostic — print key mismatch before reporting n=0 | Silent n=0 hides whether annotations loaded at all |
| E5 | Stratified TBX11K sampling (`test.txt` list) | `all_trainval.txt` hits `imgs/tb/` first → all 200 samples are TB-positive |

## Datasets to attach on Kaggle
| Dataset slug | Mount path |
|---|---|
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/datasets/iahmedhabib/medsam-vit-b/` |
| `mabdullahi454/tb-pipeline-checkpoints` | `/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/` |
| *Your trained MoE output from Notebook 2* | `/kaggle/input/<your-moe-v2-dataset>/moe_v2/` |
| `iahmedhabib/montgomery` | `/kaggle/input/datasets/iahmedhabib/montgomery/montgomery` |
| `iahmedhabib/shehzhenn` | `/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen` |
| `usmanshams/tbx-11` | `/kaggle/input/datasets/usmanshams/tbx-11/TBX11K` |
| `organizations/nih-chest-xrays` | `/kaggle/input/datasets/organizations/nih-chest-xrays/data` |

**GPU:** T4 × 1  
**Estimated runtime:** ~35 min

In [ ]:
# ── Cell 1: Clone repo and install deps ──────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/AhmedHabib00/dl-project-codebase.git"
REPO_DIR = Path("/kaggle/working/repo")
OUT_DIR  = Path("/kaggle/working/eval_moe_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements.txt")], check=True)
print("Setup complete.")

In [ ]:
# ── Cell 2: Checkpoint paths ─────────────────────────────────────────────────
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

MONTGOMERY_ROOT = KAGGLE_INPUT / "datasets/iahmedhabib/montgomery/montgomery"
SHENZHEN_ROOT   = KAGGLE_INPUT / "datasets/iahmedhabib/shehzhenn/shenzhen"
TBX11K_ROOT     = KAGGLE_INPUT / "datasets/usmanshams/tbx-11/TBX11K"
NIH_ROOT        = KAGGLE_INPUT / "datasets/organizations/nih-chest-xrays/data"

MEDSAM_CKPT  = KAGGLE_INPUT / "datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
C1_ADAPTER   = KAGGLE_INPUT / "datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors"
C4_DECODER   = KAGGLE_INPUT / "datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt"

# ---- UPDATE THIS to the output directory from train_moe_sprint3_v2.ipynb ----
MOE_SAVE_DIR = KAGGLE_INPUT / "<your-moe-v2-dataset>/moe_v2"  # <-- UPDATE THIS
MOE_BEST_PT  = MOE_SAVE_DIR / "moe_best.pt"

def check(label, path, required=True):
    status = "OK" if path.exists() else "MISSING"
    flag   = "[REQUIRED]" if required else "[optional]"
    print(f"  {flag} {label:<45}: {status}")
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found at {path}")

check("MedSAM checkpoint",  MEDSAM_CKPT)
check("C1 LoRA adapter",    C1_ADAPTER)
check("C4 lung decoder",    C4_DECODER)
check("MoE best.pt",        MOE_BEST_PT)
check("Montgomery dataset", MONTGOMERY_ROOT)
check("Shenzhen dataset",   SHENZHEN_ROOT)
check("TBX11K dataset",     TBX11K_ROOT)
check("NIH-CXR14 dataset",  NIH_ROOT, required=False)
print("All required mounts present.")

In [ ]:
# ── Cell 3: Build override configs ───────────────────────────────────────────
import yaml
from pathlib import Path

moe_yaml_src = REPO_DIR / "configs" / "moe.yaml"
with moe_yaml_src.open() as f:
    moe_cfg = yaml.safe_load(f)

moe_cfg["component1"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component1"]["adapter_path"]             = str(C1_ADAPTER)
moe_cfg["component4"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component4"]["decoder_checkpoint_path"] = str(C4_DECODER)
moe_cfg["moe"]["checkpoint_path"]                = str(MOE_BEST_PT)

moe_file = OUT_DIR / "moe_eval.yaml"
with moe_file.open("w") as f:
    yaml.safe_dump(moe_cfg, f, sort_keys=False)

paths_cfg = {
    "project_root": str(REPO_DIR),
    "datasets": {
        "montgomery": str(MONTGOMERY_ROOT),
        "shenzhen":   str(SHENZHEN_ROOT),
        "tbx11k":     str(TBX11K_ROOT),
        "nih_cxr14":  str(NIH_ROOT) if NIH_ROOT.exists() else "",
    },
}
paths_file = OUT_DIR / "paths_eval.yaml"
with paths_file.open("w") as f:
    yaml.safe_dump(paths_cfg, f, sort_keys=False)

print("Configs written.")
print(f"  MoE checkpoint : {moe_cfg['moe']['checkpoint_path']}")
print(f"  C1 adapter     : {moe_cfg['component1']['adapter_path']}")

In [ ]:
# ── Cell 4: TBX11K bbox diagnostic — run BEFORE evaluation ──────────────────
# Diagnoses the silent "moe_lesion_dice n=0" failure.
# Expected: tbx_boxes has 1000+ keys, and their format matches image_basename.

import sys
sys.path.insert(0, str(REPO_DIR))
from src.evaluation.baseline_eval import load_tbx11k_bbox_index

tbx_boxes = load_tbx11k_bbox_index(TBX11K_ROOT)
print(f"TBX11K bbox index loaded: {len(tbx_boxes)} unique images with annotations")

if tbx_boxes:
    sample_keys = list(tbx_boxes.keys())[:5]
    print(f"  Key format sample: {sample_keys}")
    print("  (keys should be bare filenames like '00001.png', not full paths)")
    # Also check actual image files in TBX11K
    import os
    sample_files = []
    for root, dirs, files in os.walk(TBX11K_ROOT / "imgs"):
        for fname in files[:3]:
            sample_files.append(fname)
        if len(sample_files) >= 5:
            break
    print(f"  Actual TBX11K image basenames: {sample_files}")
    # Check overlap
    keys_set = set(tbx_boxes.keys())
    matches = sum(1 for f in sample_files if f in keys_set)
    print(f"  Sample overlap check: {matches}/{len(sample_files)} sample files found in bbox index")
else:
    print("  WARNING: bbox index is empty — annotations/json/*.json may not exist at this path.")
    print(f"  Checked: {TBX11K_ROOT / 'annotations' / 'json'}")

In [ ]:
# ── Cell 5: Run MoE evaluation ───────────────────────────────────────────────
# limit_per_domain=200 with stratified TBX sampling (test.txt avoids train leakage)
import sys
sys.path.insert(0, str(REPO_DIR))

from src.evaluation.moe_eval import run_moe_evaluation

LIMIT_PER_DOMAIN = 200

summary = run_moe_evaluation(
    moe_config_path  = moe_file,
    paths_config_path= paths_file,
    output_dir       = OUT_DIR,
    limit_per_domain = LIMIT_PER_DOMAIN,
    tbx_list_name    = "all_trainval.txt",   # use test split only (avoids bbox annotation overlap)
    repo_root        = REPO_DIR,
)
print("\nMoE evaluation complete.")

In [ ]:
# ── Cell 6: Component metrics ─────────────────────────────────────────────────
import pandas as pd

comp_df = pd.read_csv(OUT_DIR / "moe_components.csv")
comp_df["value"] = comp_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE COMPONENT-LEVEL METRICS ===")
print(comp_df[["metric","dataset","value","n","notes"]].to_string(index=False))

In [ ]:
# ── Cell 7: System metrics ────────────────────────────────────────────────────
import pandas as pd

sys_df = pd.read_csv(OUT_DIR / "moe_system.csv")
sys_df["value"] = sys_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE SYSTEM-LEVEL METRICS ===")
print(sys_df[["metric","dataset","value","n","notes"]].to_string(index=False))

In [ ]:
# ── Cell 8: NIH ALP sanity gate ───────────────────────────────────────────────
# Experts that learned to paint lungs instead of lesions will have NIH ALP ~25%.
# After the fix, NIH ALP mean should be < 8% (only coincidental activations).
# FAIL here means the cache was still built with the old script.

import pandas as pd

pi_df  = pd.read_csv(OUT_DIR / "moe_per_image.csv")
nih_df = pi_df[pi_df["dataset_id"] == "nih_cxr14"]

if nih_df.empty:
    print("NIH-CXR14 was not included in this eval run (NIH dataset not attached).")
    print("Attach the NIH dataset and rerun to use this sanity gate.")
else:
    alp_mean = nih_df["alp"].mean()
    alp_p95  = nih_df["alp"].quantile(0.95)
    ok = alp_mean < 8.0
    print(f"NIH-CXR14 ALP (n={len(nih_df)}):")
    print(f"  mean : {alp_mean:.2f}%  ({'OK — experts not painting lungs' if ok else 'FAIL — rebuild cache with nb_fix_cache_rebuild.ipynb'})")
    print(f"  p95  : {alp_p95:.2f}%")
    if not ok:
        print()
        print("  ACTION: The cache was likely built with the old broken script.")
        print("  Run nb_fix_cache_rebuild.ipynb to rebuild, then retrain and re-eval.")

In [ ]:
# ── Cell 9: Bootstrap 95% CI on Timika AUROC (all datasets) ──────────────────
# Montgomery has n=28, 14 TB+ — raw AUROC is unreliable without CI.
# A result of 0.34 ± 0.20 (CI: 0.14–0.54) is very different from 0.74 ± 0.07.

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

pi_df = pd.read_csv(OUT_DIR / "moe_per_image.csv")

def bootstrap_auroc(y_true, y_score, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auc_vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        ys = y_score[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            auc_vals.append(roc_auc_score(yt, ys))
        except Exception:
            pass
    if not auc_vals:
        return float("nan"), float("nan"), float("nan")
    return float(np.mean(auc_vals)), float(np.percentile(auc_vals, 2.5)), float(np.percentile(auc_vals, 97.5))

print("Timika AUROC with 95% bootstrap CI (2000 resamples):")
print(f"  {'Dataset':<14} {'AUROC':>8}  {'95% CI':>16}  {'n_total':>8}  {'n_TB+':>7}")
print("  " + "-"*62)

TARGET = {"montgomery": 0.60, "shenzhen": 0.70, "tbx11k": 0.70}

for dom in ["montgomery", "shenzhen", "tbx11k"]:
    sub = pi_df[(pi_df["dataset_id"] == dom) & pi_df["tb_label"].notna()].copy()
    if sub.empty or sub["tb_label"].nunique() < 2:
        print(f"  {dom:<14}   N/A      (no/single-class labels, n={len(sub)})")
        continue
    y_true  = sub["tb_label"].astype(int).values
    y_score = sub["timika_score"].values
    mean_auc, lo, hi = bootstrap_auroc(y_true, y_score)
    n_pos = int(y_true.sum())
    target = TARGET.get(dom, 0.70)
    met = "PASS" if mean_auc >= target else "FAIL"
    print(f"  {dom:<14}  {mean_auc:>6.4f}   [{lo:.4f} – {hi:.4f}]  {len(sub):>8}  {n_pos:>7}  [{met} (target≥{target})]")

print()
print("Note: Montgomery with n<30 and narrow TB class balance will always have wide CI.")
print("Shenzhen (n~120+) is the primary metric to trust.")

In [ ]:
# ── Cell 10: TBX11K MoE lesion Dice (with diagnostic) ────────────────────────
import pandas as pd

comp_df = pd.read_csv(OUT_DIR / "moe_components.csv")
dice_row = comp_df[comp_df["metric"] == "moe_lesion_dice"]

if dice_row.empty:
    print("moe_lesion_dice not found in components CSV.")
elif dice_row["value"].isna().all():
    print("moe_lesion_dice = N/A")
    print(f"  Notes: {dice_row['notes'].values[0] if not dice_row.empty else 'none'}")
    print()
    print("Diagnosing bbox match failure:")
    pi_df = pd.read_csv(OUT_DIR / "moe_per_image.csv")
    tbx_pi = pi_df[pi_df["dataset_id"] == "tbx11k"]
    print(f"  TBX11K images in per-image CSV: {len(tbx_pi)}")
    if not tbx_pi.empty:
        print(f"  Sample image_basenames: {tbx_pi['image_basename'].head(5).tolist()}")
    
    from src.evaluation.baseline_eval import load_tbx11k_bbox_index
    tbx_boxes = load_tbx11k_bbox_index(TBX11K_ROOT)
    print(f"  bbox_index keys loaded: {len(tbx_boxes)}")
    if tbx_boxes:
        print(f"  Sample bbox_index keys: {list(tbx_boxes.keys())[:5]}")
        overlap = set(tbx_pi["image_basename"].values) & set(tbx_boxes.keys())
        print(f"  Overlap (images in eval that have GT): {len(overlap)}")
        if len(overlap) == 0:
            print("  KEY FORMAT MISMATCH — image_basename vs bbox_index keys do not agree.")
            print("  Check whether TBX11K test.txt exists; if not, fallback to all_trainval.txt")
else:
    val = float(dice_row["value"].values[0])
    n   = int(dice_row["n"].values[0])
    print(f"moe_lesion_dice (TBX11K): {val:.4f}  (n={n} images)")
    target_dice = 0.30
    status = "PASS" if val >= target_dice else "Below target"
    print(f"  Target ≥ {target_dice}: {status}")

In [ ]:
# ── Cell 11: Expert routing weight distribution ────────────────────────────────
# After fix, routing should NOT be uniform. Expect clear specialisation:
#  - TBX11K active-TB images → consolidation + fibrosis weights dominant
#  - NIH healthy images → all weights near 0.25 (no clear expert wins = model uncertain)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pi_df = pd.read_csv(OUT_DIR / "moe_per_image.csv")
expert_cols  = ["routing_consolidation", "routing_cavity", "routing_fibrosis", "routing_nodule"]
expert_labels= ["Consolidation", "Cavity", "Fibrosis", "Nodule"]

# Check which columns exist
available = [c for c in expert_cols if c in pi_df.columns]
if not available:
    print("Routing columns not found in per-image CSV. Check moe_eval.py outputs.")
else:
    datasets = [d for d in ["montgomery", "shenzhen", "tbx11k"] if d in pi_df["dataset_id"].values]
    n_ds = len(datasets)
    if n_ds == 0:
        print("No dataset results found.")
    else:
        fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
        if n_ds == 1:
            axes = [axes]
        colors = ["#2196F3", "#f44336", "#FF9800", "#9C27B0"]
        for ax, dom in zip(axes, datasets):
            sub = pi_df[(pi_df["dataset_id"] == dom)][available].dropna()
            means = sub.mean()
            stds  = sub.std()
            labels_avail = [expert_labels[expert_cols.index(c)] for c in available]
            bars = ax.bar(labels_avail, means, yerr=stds, capsize=4,
                          color=colors[:len(available)], alpha=0.85)
            ax.axhline(0.25, color="gray", linestyle="--", linewidth=1, label="Uniform (0.25)")
            ax.set_title(f"{dom}\n(n={len(sub)})", fontsize=11)
            ax.set_ylabel("Mean routing weight")
            ax.set_ylim(0, 0.7)
            ax.legend(fontsize=8)
        plt.suptitle("Expert Routing Weights\n(deviation from 0.25 = specialisation)", fontsize=12)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "routing_weights.png", dpi=120)
        plt.show()
        print("Saved: routing_weights.png")

In [ ]:
# ── Cell 12: Sprint 3 target table ───────────────────────────────────────────
import pandas as pd
import json

sys_df = pd.read_csv(OUT_DIR / "moe_system.csv")
sys_df["value"] = pd.to_numeric(sys_df["value"], errors="coerce")

TARGETS = {
    ("timika_auroc", "montgomery"): ("≥ 0.60", 0.60),
    ("timika_auroc", "shenzhen"):   ("≥ 0.70", 0.70),
    ("tb_head_auroc", "shenzhen"):  ("≥ 0.85", 0.85),
    ("tb_head_auroc", "montgomery"):("≥ 0.80", 0.80),
}

print(f"{'Metric':<25} {'Dataset':<14} {'Achieved':>10}  {'Target':>8}  {'Status'}")
print("-" * 70)
for (metric, dataset), (target_str, target_val) in TARGETS.items():
    row = sys_df[(sys_df["metric"] == metric) & (sys_df["dataset"] == dataset)]
    if row.empty or pd.isna(row["value"].values[0]):
        achieved = "N/A"
        status = "N/A"
    else:
        val = float(row["value"].values[0])
        achieved = f"{val:.4f}"
        status = "PASS" if val >= target_val else f"FAIL (gap={target_val-val:.4f})"
    print(f"{metric:<25} {dataset:<14} {achieved:>10}  {target_str:>8}  {status}")

print()
print("Note: Montgomery AUROC has very high variance (n≈28, 14 TB+).")
print("Primary target is Shenzhen timika_auroc ≥ 0.70.")
print("See Cell 9 for bootstrap 95% CI before reporting any Montgomery result.")